# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matiasdibe23-sys/Flyrank-Starter-Notebooks/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Refreshing content leads to a significant increase in organic traffic (30%+ CTR improvement).
* **Where does the label come from?** The target variable appears to be constructed by comparing post-refresh organic clicks against a pre-refresh historical average across selected pages.
* **Methodology Questions:**
  1. *Label Construction:* Is the baseline period controlled for seasonality, Google algorithm updates, or sitewide domain growth, or are we attributing macro-traffic gains entirely to the refresh action?
  2. *Validation Design:* Was the validation evaluated on unseen domains/clients via a grouped split (`GroupKFold`), or were pages from the same website present in both training and test sets? If the latter, the model may be learning client-specific domain authority rather than true content refresh impact.

### Finding 2: AI-driven features reliably predict page traffic decay prior to rank drops.
* **Where does the label come from?** The decay target is derived from a drop in short-term click volume (e.g., $H_2$ vs. $H_1$ click ratio below a threshold like 80%).
* **Methodology Questions:**
  1. *Selection Bias & Mean Reversion:* Does the decay label filter for pages with temporary high traffic spikes in $H_1$? Pages with an unusually high $H_1$ naturally regress to the mean in $H_2$ without representing actual SEO decay.
  2. *Evaluation Scope:* Was the performance evaluated across clients of varying scale, or do large clients with high click volumes skew the precision/recall metrics?

In [ ]:
# Section 1 check: a SIMULATION (not FlyRank data) showing why a decay label picks up mean reversion
import numpy as np
import pandas as pd

print("=== SIMULATION: mean reversion with no real decay ===")
np.random.seed(42)
h1_clicks = np.random.negative_binomial(10, 0.1, 1000)
# H2 is independent of H1 with the same mean, so nothing "decays" by construction
h2_clicks = np.random.poisson(h1_clicks.mean(), 1000)

df_sim = pd.DataFrame({"h1": h1_clicks, "h2": h2_clicks})
top_h1 = df_sim[df_sim["h1"] > df_sim["h1"].quantile(0.80)]
decay_rate = (top_h1["h2"] < top_h1["h1"] * 0.80).mean()

print(f"Total simulated pages: {len(df_sim)}")
print(f"Top 20% traffic pages in H1: {len(top_h1)}")
print(f"Apparent 'decay' rate in the top H1 pages, from mean reversion alone: {decay_rate:.2%}")

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Model evaluation: naive split vs honest splits

* **Before (naive stratified split, W05 v1):**
  * **Setup:** random stratified split across all pages, and a label defined by a SQL rule over the same columns used as features (impressions > 1000, CTR < 0.015, position <= 20).
  * **Result:** Decision Tree and Random Forest reached **1.0000** on ROC-AUC, precision and recall.
  * **Diagnosis:** target leakage by construction. A shallow tree re-derives the rule. The number says nothing about prediction (see `work/outputs/w05_audit_report.md`).

* **After 1 (W05 v2, one client-grouped split, June 2026, raw label):** 4,439 pages, 6 test clients never seen in training. The label is "clicks/day in days 16-30 < 80% of days 1-15". Test prevalence 0.6315, so a random scorer has a PR-AUC of about 0.63.
  * Click-momentum rule 0.675 ROC-AUC, Random Forest 0.612, logistic regression 0.557, decision tree 0.524, Week-4 score 0.418. The forest's ROC-AUC inside the training clients (GroupKFold) was 0.590 +/- 0.041.
  * With shuffled labels the forest scored 0.518, close to the 0.5 expected.

* **After 2 (capstone, time-forward and client-aware, market-adjusted label):** train January-April 2026, test May-June; client-bootstrap intervals.
  * Seen clients (19 clients, 1,870 pages): Random Forest 0.704 against 0.633 for the momentum rule; difference +0.071, 95% interval +0.015 to +0.092. Week-4 score 0.431.
  * Unseen clients (6 clients, 433 pages): Random Forest 0.714, momentum 0.702; difference +0.011, interval -0.014 to +0.446. The data cannot separate them.

* **Honest reading:**
  * The one-line momentum rule is a strong baseline. A Random Forest beats it modestly on seen clients in a time-forward test, and not on the single June split or on unseen clients.
  * The Week-4 score ranks below chance in every test, so it should not be used to flag pages at risk.
  * I do not have evidence for *why* the forest does worse on the single June split. With only 6 test clients that result is noisy, so I do not claim overfitting.

In [ ]:
# Section 2 code: the before/after table, from saved outputs (no model is re-trained here)
import json, os
import pandas as pd

# Transcribed from the saved output of w05_model.ipynb, section 3 (raw label, 6 unseen test clients, prevalence 0.6315)
w05_single_split = pd.DataFrame({
    "Scorer": ["Click-momentum rule", "Random Forest", "Logistic regression", "Decision tree (depth 4)", "RF prob x H1 clicks", "Week-4 heuristic"],
    "ROC-AUC": [0.6751, 0.6123, 0.5565, 0.5243, 0.5027, 0.4176],
    "PR-AUC": [0.7665, 0.7228, 0.6659, 0.6742, 0.6412, 0.6044],
})
print("=== W05 v1 (naive split, deterministic label): ROC-AUC 1.0000 -> leakage, not a result ===")
print("\n=== W05 v2: one client-grouped split, June 2026 (random scorer's PR-AUC is about 0.63) ===")
print(w05_single_split.to_string(index=False))

path = "work/outputs/capstone_results.json"
if os.path.exists(path):
    cap = json.load(open(path))
    for tag in ("Test A", "Test B"):
        b = pd.DataFrame(cap["bootstrap_auc"][tag]["scorers"]).T
        b = b.loc[["Momentum rule", "Random Forest", "Gradient boosting", "Week-4 score", "Size only (H1 clicks)"], ["ROC-AUC", "CI low", "CI high", "vs momentum", "diff CI low", "diff CI high"]]
        print(f"\n=== Capstone {tag}: May+June, {cap['bootstrap_auc'][tag]['clients']} clients, client-bootstrap 95% intervals ===")
        print(b.to_string())
else:
    print(f"\n[missing] {path}: run capstone.ipynb first to print the time-forward table.")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Feature leakage audit and failure examples

1. **Feature leakage check:**
   * Every feature is computed from days 1-15 of the month; the label comes only from days 16 to the end. The code below asserts it for the real feature lists of W05 and of the capstone.
   * A deliberate-leak control shows the harness can detect a leak: adding the label-window ratio as a feature raised the capstone Random Forest from 0.704 to 0.946 ROC-AUC, and adding that ratio divided by the month's median (the label's own quantity) gave 1.000.
   * Shuffled labels collapse the score to 0.518 (W05) and 0.506 (capstone), as they should.

2. **Failure examples (real, from the saved output of `w05_model.ipynb`, section 4; no ids):**
   * **False positives (predicted decay, did not decay):** the ten most confident ones are small pages (20-41 clicks in days 1-15) with scores 0.71-0.76 and a click momentum of 0.49-0.83. Their clicks were already falling inside the first half, and then their second-half rate stayed above 80% of the first-half rate. Small counts make this kind of swing common; I did not test the cause beyond that.
   * **False negatives (decayed, model gave a low score):** the four clearest misses printed had a *rising* momentum (1.24-1.68) and then fell, which looks like mean reversion after a good week. This is a reading of four rows, not a tested explanation.
   * **At queue level (capstone, May 2026):** 4 of the top 10 pages decayed. Four of the six that did not were flagged only HIGH_VOLUME (size, not falling clicks). The size term pulls big pages to the top.

In [ ]:
# Section 3 code: leakage assertions on the real feature lists, plus the real failure examples
import json, os
import pandas as pd

W05_FEATURES = ["impr_h1", "clicks_h1", "ctr_h1", "pos_h1", "pv_h1", "pv_missing", "days_h1", "click_momentum"]
CAPSTONE_FEATURES = ["impr_h1", "ctr_h1", "pos_h1", "click_momentum", "impr_momentum", "pos_slip"]
LABEL_WINDOW_COLS = {"clicks_h2", "days_h2", "rate_h2", "ratio", "target_decay", "clicks_lost_h2",
                     "y_adj", "y_raw", "lost_adj", "lost_raw", "cohort_ratio", "n_days_h2_cal"}

for name, feats in (("W05", W05_FEATURES), ("capstone", CAPSTONE_FEATURES)):
    assert LABEL_WINDOW_COLS.isdisjoint(feats), f"LEAKAGE: a label-window column is a {name} feature"
    assert not any("h2" in f for f in feats), f"LEAKAGE: a {name} feature name refers to the second half"
print("[PASSED] No label-window column is a feature in W05 or in the capstone.")

path = "work/outputs/capstone_results.json"
if os.path.exists(path):
    nc = json.load(open(path))["leakage_negative_control"]
    print("Negative control (Test A ROC-AUC):", {k: round(v, 3) for k, v in nc.items()})

# Transcribed from w05_model.ipynb, section 4 output (no ids): the 10 most confident false positives
fp = pd.DataFrame({
    "score": [0.7637, 0.7550, 0.7545, 0.7477, 0.7415, 0.7378, 0.7371, 0.7271, 0.7138, 0.7112],
    "clicks_h1": [26, 26, 26, 25, 22, 28, 20, 25, 23, 41],
    "click_momentum": [0.5682, 0.7557, 0.8333, 0.5882, 0.8289, 0.5721, 0.5170, 0.7159, 0.6364, 0.4861],
    "pos_h1": [5.52, 3.06, 2.60, 6.28, 32.04, 5.57, 4.17, 2.38, 5.93, 4.49],
})
print("\n=== 10 MOST CONFIDENT FALSE POSITIVES (predicted decay, did not decay) ===")
print(fp.to_string(index=False))
# First 4 of the 10 clearest misses printed in the same output
miss = pd.DataFrame({
    "score": [0.3472, 0.3527, 0.3592, 0.3607],
    "clicks_h1": [22, 20, 58, 43],
    "click_momentum": [1.6154, 1.6771, 1.5817, 1.2396],
    "pos_h1": [21.27, 13.64, 21.03, 25.22],
})
print("\n=== 4 OF THE CLEAREST MISSES (decayed, model gave a low score) ===")
print(miss.to_string(index=False))

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite (public-safe language)

* **Original bold claim (overstated):**
  > *"Our Random Forest machine learning model predicts organic traffic decay and SEO rank drops with 100% accuracy, enabling automated content refreshes before traffic falls."*
  >
  > This is what W05 v1's 1.0000 would have seemed to support. It was leakage.

* **Rewritten safe claim:**
  > *"In a time-forward test (trained on January-April 2026, tested on May-June) on 19 clients seen in training, a Random Forest ranked pages that lost clicks beyond the typical page modestly better than a one-line click-momentum rule (ROC-AUC 0.704 against 0.633; difference +0.071, 95% client-bootstrap interval +0.015 to +0.092). On 6 unseen clients the two could not be told apart. This is observational and directional evidence, useful as a decision-support signal for which pages to review first; it does not show that refreshing a page recovers clicks."*

In [ ]:
# Section 4 code: terminology check on the actual rewritten claim, and a traceability check of its numbers
import json, os

rewritten_claim = (
    "In a time-forward test (trained on January-April 2026, tested on May-June) on 19 clients seen in training, "
    "a Random Forest ranked pages that lost clicks beyond the typical page modestly better than a one-line click-momentum rule "
    "(ROC-AUC 0.704 against 0.633; difference +0.071, 95% client-bootstrap interval +0.015 to +0.092). "
    "On 6 unseen clients the two could not be told apart. This is observational and directional evidence, useful as a "
    "decision-support signal for which pages to review first; it does not show that refreshing a page recovers clicks."
)
forbidden_words = ["100%", "perfect", "guaranteed", "predicts with precision", "flawless", "significant", "proves", "causes"]
safe_words = ["observational", "directional", "decision-support"]

text = rewritten_claim.lower()
assert not any(w in text for w in forbidden_words), "Claim contains hype or causal language!"
assert all(w in text for w in safe_words), "Claim lacks public-safe terminology!"
print("[PASSED] Terminology: no hype or causal words; safe terms present.")

path = "work/outputs/capstone_results.json"
if os.path.exists(path):
    a = json.load(open(path))["bootstrap_auc"]["Test A"]["scorers"]
    assert round(a["Random Forest"]["ROC-AUC"], 3) == 0.704 and round(a["Momentum rule"]["ROC-AUC"], 3) == 0.633
    assert round(a["Random Forest"]["vs momentum"], 3) == 0.071
    assert round(a["Random Forest"]["diff CI low"], 3) == 0.015 and round(a["Random Forest"]["diff CI high"], 3) == 0.092
    print("[PASSED] Traceability: the numbers in the claim match capstone_results.json.")
else:
    print(f"[skipped] {path} not found: run capstone.ipynb to check the numbers.")

## Self-check

Confirming each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.